# 11.13 Distributional RL (C51, QR-DQN)
Distributional RL keeps the spread of possible returns, not just their average.

Distributional RL belongs in Part 11 because RL predictions change the data that arrives next. Here we keep the whole discounted-return distribution so risk, rare failures, and equal-mean alternatives remain visible. Save a copy to Drive to edit.

In [ ]:

import math
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np

SEED = 1113
rng = np.random.default_rng(SEED)
random.seed(SEED)

ACTIONS = np.array([
    [-1, 0],
    [1, 0],
    [0, -1],
    [0, 1],
])
ACTION_NAMES = np.array(["U", "D", "L", "R"])

@dataclass
class LadderEnv:
    name: str
    height: int
    width: int
    start: tuple
    goal: tuple
    walls: tuple
    slip: float
    wind: float
    step_cost: float
    goal_reward: float
    traps: dict
    max_steps: int
    continuous: bool = False

    @property
    def n_states(self):
        return self.height * self.width

    @property
    def n_actions(self):
        return len(ACTIONS)

    def state_index(self, pos):
        row, col = pos
        return row * self.width + col

    def index_state(self, idx):
        row = idx // self.width
        col = idx % self.width
        return (row, col)

    def reset(self):
        return self.state_index(self.start)

    def move(self, pos, action, local_rng):
        actual = int(action)
        if local_rng.random() < self.slip:
            actual = int(local_rng.integers(0, self.n_actions))
        delta = ACTIONS[actual].copy()
        if self.wind > 0.0 and local_rng.random() < self.wind:
            delta = delta + np.array([-1, 0])
        nxt = (pos[0] + int(delta[0]), pos[1] + int(delta[1]))
        bad_row = nxt[0] < 0 or nxt[0] >= self.height
        bad_col = nxt[1] < 0 or nxt[1] >= self.width
        if bad_row or bad_col or nxt in self.walls:
            nxt = pos
        return nxt

    def step(self, state, action, local_rng):
        pos = self.index_state(int(state))
        nxt = self.move(pos, action, local_rng)
        reward = self.step_cost
        done = False
        if nxt in self.traps:
            reward = reward + float(self.traps[nxt])
        if nxt == self.goal:
            reward = reward + self.goal_reward
            done = True
        return self.state_index(nxt), reward, done


def make_rl_ladder(continuous=False):
    envs = []
    envs.append(LadderEnv("D1 two-state chain", 1, 2, (0, 0), (0, 1), tuple(), 0.0, 0.0, 0.0, 1.0, {}, 4, continuous))
    envs.append(LadderEnv("D2 slippery 3-state", 1, 3, (0, 0), (0, 2), tuple(), 0.15, 0.0, -0.01, 1.0, {}, 8, continuous))
    envs.append(LadderEnv("D3 4x4 gridworld", 4, 4, (3, 0), (0, 3), ((1, 1),), 0.05, 0.0, -0.02, 1.0, {(2, 2): -0.25}, 24, continuous))
    envs.append(LadderEnv("D4 windy stochastic grid", 5, 5, (4, 0), (0, 4), ((1, 1), (2, 1), (3, 3)), 0.12, 0.18, -0.025, 1.1, {(2, 3): -0.4}, 35, continuous))
    envs.append(LadderEnv("D5 sparse reward grid", 6, 6, (5, 0), (0, 5), ((1, 1), (1, 2), (2, 2), (3, 4), (4, 1)), 0.18, 0.20, -0.03, 1.5, {(2, 4): -0.6, (4, 4): -0.3}, 50, continuous))
    return envs


def softmax(logits):
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exp_logits = np.exp(shifted)
    return exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)


def discounted_returns(rewards, gamma):
    returns = np.zeros(len(rewards), dtype=float)
    running = 0.0
    for t in range(len(rewards) - 1, -1, -1):
        running = float(rewards[t]) + gamma * running
        returns[t] = running
    return returns


def rollout(env, logits, local_rng, gamma=0.9):
    states = []
    actions = []
    rewards = []
    state = env.reset()
    for _ in range(env.max_steps):
        probs = softmax(logits[state])
        action = int(local_rng.choice(env.n_actions, p=probs))
        next_state, reward, done = env.step(state, action, local_rng)
        states.append(state)
        actions.append(action)
        rewards.append(reward)
        state = next_state
        if done:
            break
    returns = discounted_returns(np.array(rewards), gamma)
    return np.array(states), np.array(actions), np.array(rewards), returns


def evaluate_policy(env, logits, episodes=20, gamma=0.9):
    values = []
    local_rng = np.random.default_rng(SEED + env.n_states)
    for _ in range(episodes):
        _, _, rewards, _ = rollout(env, logits, local_rng, gamma)
        values.append(float(np.sum(rewards)))
    return float(np.mean(values))


def value_iteration(env, gamma=0.9, iterations=120):
    values = np.zeros(env.n_states)
    local_rng = np.random.default_rng(SEED)
    for _ in range(iterations):
        new_values = values.copy()
        for state in range(env.n_states):
            pos = env.index_state(state)
            if pos == env.goal:
                continue
            q_values = []
            for action in range(env.n_actions):
                next_state, reward, done = env.step(state, action, local_rng)
                q_values.append(reward + gamma * values[next_state] * (1.0 - float(done)))
            new_values[state] = np.max(q_values)
        values = new_values
    return values


def greedy_logits_from_values(env, values, gamma=0.9, scale=5.0):
    logits = np.zeros((env.n_states, env.n_actions))
    local_rng = np.random.default_rng(SEED + 7)
    for state in range(env.n_states):
        for action in range(env.n_actions):
            next_state, reward, done = env.step(state, action, local_rng)
            logits[state, action] = scale * (reward + gamma * values[next_state] * (1.0 - float(done)))
    return logits


def plot_policy_panel(ax, env, values, logits, title):
    grid = values.reshape(env.height, env.width)
    ax.imshow(grid, cmap="viridis")
    probs = softmax(logits)
    for state in range(env.n_states):
        row, col = env.index_state(state)
        if (row, col) in env.walls:
            ax.text(col, row, "#", ha="center", va="center", color="white")
            continue
        best = int(np.argmax(probs[state]))
        ax.text(col, row, ACTION_NAMES[best], ha="center", va="center", color="white")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])


def summarize_ladder(envs):
    for env in envs:
        print(env.name, "states", env.n_states, "actions", env.n_actions, "slip", env.slip, "wind", env.wind)
        print("start", env.start, "goal", env.goal, "walls", len(env.walls), "traps", env.traps)


def train_reinforce(env, episodes=80, gamma=0.9, lr=0.08, baseline=True):
    logits = np.zeros((env.n_states, env.n_actions))
    value = np.zeros(env.n_states)
    curve = []
    local_rng = np.random.default_rng(SEED + env.n_states)
    for _ in range(episodes):
        states, actions, rewards, returns = rollout(env, logits, local_rng, gamma)
        if len(states) == 0:
            continue
        advantages = returns.copy()
        if baseline:
            advantages = returns - value[states]
            for state, target in zip(states, returns):
                value[state] = value[state] + 0.15 * (target - value[state])
        for state, action, advantage in zip(states, actions, advantages):
            probs = softmax(logits[state])
            grad = -probs
            grad[action] = grad[action] + 1.0
            logits[state] = logits[state] + lr * advantage * grad
        curve.append(float(np.sum(rewards)))
    return logits, value, np.array(curve)


def train_actor_critic(env, episodes=80, gamma=0.9, actor_lr=0.05, critic_lr=0.12, normalize=True):
    logits = np.zeros((env.n_states, env.n_actions))
    value = np.zeros(env.n_states)
    curve = []
    local_rng = np.random.default_rng(SEED + 2 * env.n_states)
    for _ in range(episodes):
        states = []
        actions = []
        deltas = []
        rewards = []
        state = env.reset()
        for _ in range(env.max_steps):
            probs = softmax(logits[state])
            action = int(local_rng.choice(env.n_actions, p=probs))
            next_state, reward, done = env.step(state, action, local_rng)
            target = reward + gamma * value[next_state] * (1.0 - float(done))
            delta = target - value[state]
            value[state] = value[state] + critic_lr * delta
            states.append(state)
            actions.append(action)
            deltas.append(delta)
            rewards.append(reward)
            state = next_state
            if done:
                break
        advantages = np.array(deltas)
        if normalize and len(advantages) > 1:
            advantages = (advantages - np.mean(advantages)) / (np.std(advantages) + 1e-8)
        for state, action, advantage in zip(states, actions, advantages):
            probs = softmax(logits[state])
            grad = -probs
            grad[action] = grad[action] + 1.0
            logits[state] = logits[state] + actor_lr * advantage * grad
        curve.append(float(np.sum(rewards)))
    return logits, value, np.array(curve)


def gae_advantages(rewards, values, next_values, dones, gamma=0.9, lam=0.95):
    deltas = rewards + gamma * next_values * (1.0 - dones) - values
    adv = np.zeros_like(rewards, dtype=float)
    running = 0.0
    for t in range(len(rewards) - 1, -1, -1):
        running = deltas[t] + gamma * lam * (1.0 - dones[t]) * running
        adv[t] = running
    return deltas, adv


def train_gae_policy(env, lam=0.95, episodes=80, gamma=0.9):
    logits = np.zeros((env.n_states, env.n_actions))
    value = np.zeros(env.n_states)
    curve = []
    local_rng = np.random.default_rng(SEED + int(100 * lam) + env.n_states)
    for _ in range(episodes):
        states, actions, rewards, returns = rollout(env, logits, local_rng, gamma)
        if len(states) == 0:
            continue
        next_values = np.zeros(len(states))
        dones = np.zeros(len(states))
        for i, state in enumerate(states):
            if i + 1 < len(states):
                next_values[i] = value[states[i + 1]]
            else:
                dones[i] = 1.0
        _, advantages = gae_advantages(rewards, value[states], next_values, dones, gamma, lam)
        for state, target in zip(states, returns):
            value[state] = value[state] + 0.12 * (target - value[state])
        if len(advantages) > 1:
            advantages = (advantages - np.mean(advantages)) / (np.std(advantages) + 1e-8)
        for state, action, advantage in zip(states, actions, advantages):
            probs = softmax(logits[state])
            grad = -probs
            grad[action] = grad[action] + 1.0
            logits[state] = logits[state] + 0.05 * advantage * grad
        curve.append(float(np.sum(rewards)))
    return logits, value, np.array(curve)


def ppo_surrogate(old_probs, new_probs, actions, advantages, clip=0.2):
    chosen_old = old_probs[np.arange(len(actions)), actions]
    chosen_new = new_probs[np.arange(len(actions)), actions]
    ratios = chosen_new / np.maximum(chosen_old, 1e-8)
    unclipped = ratios * advantages
    clipped = np.clip(ratios, 1.0 - clip, 1.0 + clip) * advantages
    return ratios, np.minimum(unclipped, clipped)


def train_ppo(env, episodes=80, gamma=0.9, clip=0.2, clipped=True):
    logits = np.zeros((env.n_states, env.n_actions))
    value = np.zeros(env.n_states)
    curve = []
    local_rng = np.random.default_rng(SEED + 3 * env.n_states)
    for _ in range(episodes):
        states, actions, rewards, returns = rollout(env, logits, local_rng, gamma)
        if len(states) == 0:
            continue
        old_probs = softmax(logits[states])
        advantages = returns - value[states]
        if len(advantages) > 1:
            advantages = (advantages - np.mean(advantages)) / (np.std(advantages) + 1e-8)
        for _epoch in range(3):
            new_probs = softmax(logits[states])
            ratios, weights = ppo_surrogate(old_probs, new_probs, actions, advantages, clip)
            if not clipped:
                weights = ratios * advantages
            for state, action, weight in zip(states, actions, weights):
                probs = softmax(logits[state])
                grad = -probs
                grad[action] = grad[action] + 1.0
                logits[state] = logits[state] + 0.04 * weight * grad
        for state, target in zip(states, returns):
            value[state] = value[state] + 0.15 * (target - value[state])
        curve.append(float(np.sum(rewards)))
    return logits, value, np.array(curve)


def distributional_backup(rewards, gamma):
    atoms = discounted_returns(np.array(rewards, dtype=float), gamma)
    probs = np.ones_like(atoms) / len(atoms)
    return atoms, probs, float(atoms[0])


def train_distributional(env, atoms=21, episodes=60, gamma=0.9):
    support = np.linspace(-1.0, 2.0, atoms)
    pmf = np.ones((env.n_states, env.n_actions, atoms)) / atoms
    logits = np.zeros((env.n_states, env.n_actions))
    curve = []
    errors = []
    optimal = value_iteration(env, gamma)
    local_rng = np.random.default_rng(SEED + 4 * env.n_states)
    for _ in range(episodes):
        state = env.reset()
        episode_reward = 0.0
        for _step in range(env.max_steps):
            means = np.sum(pmf[state] * support[None, :], axis=1)
            action = int(np.argmax(means + local_rng.normal(0.0, 0.03, env.n_actions)))
            next_state, reward, done = env.step(state, action, local_rng)
            next_action = int(np.argmax(np.sum(pmf[next_state] * support[None, :], axis=1)))
            shifted = reward + gamma * support * (1.0 - float(done))
            target = np.interp(support, shifted, pmf[next_state, next_action], left=0.0, right=0.0)
            if np.sum(target) <= 0.0:
                nearest = int(np.argmin(np.abs(support - reward)))
                target = np.zeros(atoms)
                target[nearest] = 1.0
            target = target / np.sum(target)
            pmf[state, action] = 0.9 * pmf[state, action] + 0.1 * target
            episode_reward = episode_reward + reward
            state = next_state
            if done:
                break
        learned = np.max(np.sum(pmf * support[None, None, :], axis=2), axis=1)
        errors.append(float(np.mean(np.abs(learned - optimal))))
        curve.append(episode_reward)
    values = np.max(np.sum(pmf * support[None, None, :], axis=2), axis=1)
    logits = greedy_logits_from_values(env, values, gamma)
    spread = np.mean(np.std(pmf * support[None, None, :], axis=2))
    return logits, values, np.array(errors), float(spread)


def continuous_features(state, action):
    return np.array([1.0, state, action, state * action, action * action])


def deterministic_actor_critic(reward, next_q, gamma):
    target = reward + gamma * next_q
    critic_prediction = 0.4
    critic_error = target - critic_prediction
    return target, critic_error


def td3_toy_update(state, reward, next_state, actor_w, q1_w, q2_w, gamma=0.9, noise=0.05):
    action = float(np.tanh(actor_w * state))
    next_action = float(np.clip(np.tanh(actor_w * next_state) + noise, -1.0, 1.0))
    q1_next = float(continuous_features(next_state, next_action) @ q1_w)
    q2_next = float(continuous_features(next_state, next_action) @ q2_w)
    target = reward + gamma * min(q1_next, q2_next)
    prediction = float(continuous_features(state, action) @ q1_w)
    td_error = target - prediction
    q1_w = q1_w + 0.05 * td_error * continuous_features(state, action)
    return action, target, td_error, q1_w


def train_td3_ladder(env, episodes=70, gamma=0.9, twin=True):
    actor_w = 0.2
    q1_w = np.array([0.0, 0.2, 0.1, 0.0, -0.05])
    q2_w = np.array([-0.02, 0.15, 0.08, 0.0, -0.08])
    curve = []
    local_rng = np.random.default_rng(SEED + 5 * env.n_states)
    for _ in range(episodes):
        state = 0.0
        total = 0.0
        for _step in range(env.max_steps):
            action = float(np.clip(np.tanh(actor_w * state) + local_rng.normal(0.0, 0.15), -1.0, 1.0))
            target_position = 1.0
            next_state = float(np.clip(state + 0.25 * action + local_rng.normal(0.0, 0.03), -1.2, 1.2))
            reward = -abs(target_position - next_state) - 0.05 * action * action
            next_action = float(np.clip(np.tanh(actor_w * next_state) + local_rng.normal(0.0, 0.05), -1.0, 1.0))
            q1_next = float(continuous_features(next_state, next_action) @ q1_w)
            q2_next = float(continuous_features(next_state, next_action) @ q2_w)
            next_value = min(q1_next, q2_next) if twin else q1_next
            target = reward + gamma * next_value
            feat = continuous_features(state, action)
            td_error_1 = target - float(feat @ q1_w)
            td_error_2 = target - float(feat @ q2_w)
            q1_w = q1_w + 0.03 * td_error_1 * feat
            q2_w = q2_w + 0.03 * td_error_2 * feat
            actor_grad = q1_w[2] + q1_w[3] * state + 2.0 * q1_w[4] * np.tanh(actor_w * state)
            actor_w = actor_w + 0.01 * actor_grad * state
            total = total + reward
            state = next_state
        curve.append(total)
    values = np.linspace(-1.0, 1.0, env.n_states)
    logits = np.tile(np.array([-actor_w, actor_w, -0.5 * actor_w, 0.5 * actor_w]), (env.n_states, 1))
    return logits, values, np.array(curve)


## The concept, built once: return distributions
The lesson object is $$Z(s,a)\stackrel{D}{=}R(s,a)+\gamma Z(s',a')$$. With rewards $[1,0,2]$ and $\gamma=0.9$, the exact return is $1+0.9\cdot0+0.9^2\cdot2=2.620$. A distributional method stores atoms and probabilities, then checks that the mean agrees with the Bellman value when the support is projected correctly.

In [ ]:

rewards = np.array([1.0, 0.0, 2.0])
gamma = 0.9
atoms, probs, first_return = distributional_backup(rewards, gamma)
lesson_target = 1.0 + gamma * 0.8
q_new = 0.4 + 0.5 * (lesson_target - 0.4)

print("atoms", atoms)
print("probabilities", probs)
print("first return", first_return)
print("lesson target", lesson_target)
print("lesson Q_new", q_new)

assert round(first_return, 3) == 2.620
assert round(lesson_target, 3) == 1.720
assert round(q_new, 3) == 1.060


The same helper can represent C51-style categorical supports or QR-DQN-style quantiles. For this tiny D1 calculation, the important invariant is that the distributional mean equals the scalar Bellman return while the spread remains available for risk analysis.

In [ ]:

mean_return = float(np.sum(atoms * probs))
spread = float(np.std(atoms))
scalar_value = float(discounted_returns(rewards, gamma)[0])

print("distribution mean", round(mean_return, 3))
print("distribution spread", round(spread, 3))
print("scalar Bellman value", round(scalar_value, 3))

assert round(scalar_value, 3) == 2.620
assert mean_return > 0.0
assert spread > 0.0


## The dataset ladder: F12 sequential-decision environments
We build the D1-D5 ladder inline, from a two-state chain to a sparse stochastic windy grid. Every rung has a start, goal, transition noise, and a small enough state space for CPU-only NumPy experiments.

In [ ]:

envs = make_rl_ladder()
summarize_ladder(envs)

for env in envs:
    sample_state = env.reset()
    sample_next, sample_reward, sample_done = env.step(sample_state, 3, np.random.default_rng(SEED))
    print(env.name, "sample", sample_state, "->", sample_next, "reward", round(sample_reward, 3), "done", sample_done)


## Run distributional backups across D1-D5
Each rung learns a small categorical return distribution. The metric is value error versus a tabular planning reference, with return spread tracked to prevent collapsing the lesson to ordinary expected value.

In [ ]:

results = []
artifacts = []
for env in envs:
    logits, values, errors, spread = train_distributional(env)
    artifacts.append((env, logits, values, errors))
    results.append((env.name, float(errors[-1]), spread))

print("rung | value_error | return_spread")
for name, error, spread in results:
    print(name, round(error, 3), round(spread, 3))


## Results visualization
The closing figure has policy/value panels for every rung and a summary value-error curve.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, (env, logits, values, errors) in zip(axes[0], artifacts):
    plot_policy_panel(ax, env, values, logits, env.name)
for ax, (env, logits, values, errors) in zip(axes[1], artifacts):
    ax.plot(errors, color="tab:blue")
    ax.set_title("error " + env.name.split()[0])
    ax.set_xlabel("episode")
    ax.set_ylabel("value error")
plt.tight_layout()


## Pitfall on D5: confusing reward with return
Immediate reward distribution is not the discounted return distribution. On D5 that mistake ignores delayed goal reward and rare traps.

In [ ]:

d5 = envs[-1]
immediate_rewards = np.array([d5.step(d5.reset(), action, np.random.default_rng(SEED))[1] for action in range(d5.n_actions)])
_, values, errors, spread = train_distributional(d5)
wrong_signal = float(np.mean(immediate_rewards))
fixed_signal = float(values[d5.reset()])

print("wrong immediate reward signal", round(wrong_signal, 3))
print("fixed discounted return signal", round(fixed_signal, 3))
print("return spread retained", round(spread, 3))
assert abs(fixed_signal - wrong_signal) > 0.01


## Evaluate it + practice
- Metric: compare return or advantage/value error against a no-skill random-policy baseline on every rung.
- Sanity check: D1 should solve first because it has the shortest horizon and no stochastic transition.
- Ablation: turn off the key stabilizer for this lesson and the D5 metric should drop or become noisier.
- Failure signals: exploding logits, a value table with impossible magnitudes, or a policy that never reaches the goal.

Practice prompts:
1. Change $\gamma$ from 0.9 to 0.7 and explain which rungs lose the most return.
2. Replace the categorical atoms with quantile locations and compare the D5 spread.
3. Plot the immediate reward histogram beside discounted return atoms for D5.


In [ ]:
# Your code here


In [ ]:
# Your code here
